In [ ]:
import pandas as pd
import arviz as az
from datetime import datetime
import json
import pycountry
from matplotlib import pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats

from emu_renewal.inputs import DATE_FORMAT, BASE_PATH, DATA_PATH, ANALYSIS_TYPES
from emu_renewal.outputs import load_last_runs_from_path, get_multianalysis_dispvals_from_idatas, load_targets

set_matplotlib_formats("svg")

In [ ]:
initial_countries = json.load(open(DATA_PATH / f"config/countries.json", "r"))
countries = initial_countries["admissions"] + initial_countries["occupancy"]

In [ ]:
analysis_name = "run_20250219_1654"

In [ ]:
spagh_dfs = {}
targets = {}
for country in countries:
    country_path = BASE_PATH / "outputs" / analysis_name / country
    country_spaghs = [pd.read_hdf(country_path / a / "spaghetti.h5") for a in ANALYSIS_TYPES]
    spagh_dfs[country] = pd.concat(country_spaghs, keys=ANALYSIS_TYPES, axis=1)
    targets[country] = load_targets(country_path / "no_mob")

In [ ]:
for c in countries:
    country_spaghs = spagh_dfs[c]
    country_targs = targets[c]
    fig, axes = plt.subplots(len(country_targs), len(ANALYSIS_TYPES), figsize=[12, 15], sharex=True, sharey="row")
    fig.suptitle(c, fontsize=18, y=1.0)
    for o, out in enumerate(country_targs):
        for a, analysis in enumerate(ANALYSIS_TYPES):
            ax = axes[o, a]
            for col in country_spaghs[analysis][out].columns:
                ax.plot(country_spaghs[analysis].index, country_spaghs[analysis][out, col], color="black", linewidth=0.2)
            target = country_targs[out]
            ax.plot(target.index, target, linewidth=0.0, marker=".")
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=70)
            if o == 0:
                ax.set_title(analysis, fontsize=10)
            if a == 0:
                ax.set_ylabel(out, fontsize=10)
        # # Ceiling in case of single run going very high at the end
        # if "prop_" not in out:
        #     axes[o, 0].set_ylim([0.0, min([country_targs[out].max() * 2.0, axes[o, 0].get_ylim()[1]])])
    fig.tight_layout()
    fig.subplots_adjust(wspace=0.05)

In [ ]:
def get_implausible_chains(targs, spagh, indicator, rel_val):
    implausible_val = targs[indicator].max() * rel_val
    implausible_runs = spagh[indicator].ge(implausible_val).any()
    return [int(chain) for chain in set([run[1] for run in implausible_runs[implausible_runs].index])]

def drop_cols_from_output_df(df, cols):
    retain_cols = [col for col in df.columns if int(col[1][1]) not in cols]
    return df[retain_cols]

new_spaghs = {}
drop_chains = {}
for analysis in ANALYSIS_TYPES:
    drop_chains[analysis] = get_implausible_chains(targs, spaghs[analysis], "weekly_cases", 5.0)
    new_spaghs[analysis] = drop_cols_from_output_df(spaghs[analysis], drop_chains[analysis])